In [1]:
import sys
import subprocess
import site
import os
import importlib

print("Çeviri motoru (deep-translator) kuruluyor...")

try:
    # deep-translator kütüphanesini kur
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'deep-translator'])
    
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
        
    importlib.invalidate_caches()
    from deep_translator import GoogleTranslator
    import pandas as pd
    from tqdm.notebook import tqdm
    
    print("✅ TEST BAŞARILI: Çeviri modülü başarıyla kuruldu!")
except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

Çeviri motoru (deep-translator) kuruluyor...
✅ TEST BAŞARILI: Çeviri modülü başarıyla kuruldu!


In [2]:
import pandas as pd
import time
import os
from deep_translator import GoogleTranslator
from tqdm.notebook import tqdm

INPUT_FILE = "data/01_pubmed_abstracts.csv"
OUTPUT_FILE = "data/01_pubmed_abstracts_tr.csv"

print("Çeviri boru hattı başlatılıyor...")

# 1. Orijinal veriyi yükle
try:
    df_orijinal = pd.read_csv(INPUT_FILE)
    toplam_kayit = len(df_orijinal)
    print(f"Orijinal PubMed dosyasında {toplam_kayit} kayıt bulundu.")
except FileNotFoundError:
    print(f"HATA: {INPUT_FILE} bulunamadı. Lütfen dosya adını kontrol edin.")
    raise

# 2. Kaldığı yerden devam etme mantığı (Resuming)
islenmis_indexler = set()
if os.path.isfile(OUTPUT_FILE):
    df_islenmis = pd.read_csv(OUTPUT_FILE)
    # Eğer aynı başlık işlenmişse atlamak için index veya başlık listesi tutalım
    islenmis_basliklar = set(df_islenmis['title'].dropna().tolist())
    print(f"Daha önce çevrilmiş {len(df_islenmis)} kayıt bulundu. Kaldığı yerden devam edilecek...")
else:
    # Dosya yoksa başlıklarla oluştur
    pd.DataFrame(columns=["source", "title", "abstract"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    islenmis_basliklar = set()

# Çevirmen objesi
translator = GoogleTranslator(source='en', target='tr')

# Çevrilecek verileri filtrele
df_cevrilecek = df_orijinal[~df_orijinal['title'].isin(islenmis_basliklar)]
kalan_kayit_sayisi = len(df_cevrilecek)

print(f"Çevrilecek bekleyen makale sayısı: {kalan_kayit_sayisi}")

# 3. Çeviri Döngüsü
yeni_veriler = []
basarili_sayac = 0

with tqdm(total=kalan_kayit_sayisi, desc="İngilizce -> Türkçe Çeviri") as pbar:
    for index, row in df_cevrilecek.iterrows():
        try:
            # Başlık ve abstract İngilizce ise çevir
            # (Metin çok uzunsa Google Translate 5000 karakter sınırına takılır, PubMed abstractları genelde 1500-3000 arasıdır, sorun olmaz)
            orj_title = str(row['title'])
            orj_abstract = str(row['abstract'])
            
            tr_title = translator.translate(orj_title)
            tr_abstract = translator.translate(orj_abstract)
            
            yeni_veriler.append({
                "source": "PubMed_TR",
                "title": tr_title,
                "abstract": tr_abstract
            })
            
            basarili_sayac += 1
            pbar.update(1)
            
            # Veri kaybını önlemek için 50 satırda bir dosyaya yaz
            if len(yeni_veriler) >= 50:
                pd.DataFrame(yeni_veriler).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
                yeni_veriler = [] # Listeyi temizle
                
            # Google'dan ban yememek için zorunlu (ama kısa) dinlenme
            time.sleep(0.5) 
            
        except Exception as e:
            # Muhtemel 429 Too Many Requests veya internet kopması hatası
            print(f"\n[UYARI] Çeviri sırasında hata oluştu. Google sunucusu bizi yavaşlatmış olabilir. 30 saniye bekleniyor... Hata detayı: {e}")
            
            # O ana kadar olan veriyi kaybetmemek için hemen diske yaz
            if yeni_veriler:
                pd.DataFrame(yeni_veriler).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
                yeni_veriler = []
                
            time.sleep(30)
            continue

# Döngü bitince kalan son partiyi de yaz
if yeni_veriler:
    pd.DataFrame(yeni_veriler).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")

print(f"\n🎉 Çeviri işlemi tamamlandı! Tamamı Türkçe olan yeni veri seti '{OUTPUT_FILE}' olarak kaydedildi.")

Çeviri boru hattı başlatılıyor...
Orijinal PubMed dosyasında 8732 kayıt bulundu.
Çevrilecek bekleyen makale sayısı: 8732


İngilizce -> Türkçe Çeviri:   0%|          | 0/8732 [00:00<?, ?it/s]


[UYARI] Çeviri sırasında hata oluştu. Google sunucusu bizi yavaşlatmış olabilir. 30 saniye bekleniyor... Hata detayı: Regional variation of underlying kidney diseases in children undergoing chronic kidney replacement therapy around the globe. --> No translation was found using the current translator. Try another translator?

[UYARI] Çeviri sırasında hata oluştu. Google sunucusu bizi yavaşlatmış olabilir. 30 saniye bekleniyor... Hata detayı: Cell free DNA and MiRNA analysis by quantitative Real-Time polymerase chain reaction in postmortem interval determination. --> No translation was found using the current translator. Try another translator?

[UYARI] Çeviri sırasında hata oluştu. Google sunucusu bizi yavaşlatmış olabilir. 30 saniye bekleniyor... Hata detayı: Analysing YouTube as a health resource: quality and reliability of videos on pediatric appendicitis. --> No translation was found using the current translator. Try another translator?

[UYARI] Çeviri sırasında hata oluştu. Google